# 03 — Bi-Encoder (Shared Weights)

**Architecture**: Single BERT instance encodes both reviews and products.
  
**Loss**: InfoNCE with in-batch random negatives  
**Pooling**: Mean pooling (default)  
**Backbone**: bert-base-uncased (110M parameters)

This is the baseline dense model. We expect improvement over BM25 on vocabulary-mismatch queries.

In [ ]:
import sys, os
sys.path.insert(0, '..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from src.encoder import BiEncoder
from src.loss import infonce_loss
from src.dense_retriever import DenseRetriever
from src.metrics import compute_metrics, aggregate, print_metrics_table

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
Path('../results').mkdir(parents=True, exist_ok=True)

In [ ]:
# Option A: Run training inline (slow, ~45 min on T4)
# Option B: Use pre-trained checkpoint (recommended in notebook)

CHECKPOINT = '../artifacts/models/biencoder_seed42/best_model'
TRAINED = os.path.exists(CHECKPOINT)
print(f'Checkpoint exists: {TRAINED}')

if not TRAINED:
    print('\nRun this first:')
    print('  !python train.py --model-type biencoder --neg-mode random --output-dir artifacts/models/biencoder_seed42/')

In [ ]:
# Load corpus and test set
corpus_df = pd.read_parquet('../data/corpus.parquet')
test_df   = pd.read_parquet('../data/test.parquet')

corpus_ids  = corpus_df['product_id'].tolist()
corpus_docs = corpus_df['product_doc'].tolist()
query_texts = test_df['review_text'].tolist()

print(f'Corpus: {len(corpus_df):,} | Test: {len(test_df):,}')

In [ ]:
# Zero-shot evaluation (no fine-tuning)
print('\n=== ZERO-SHOT EVALUATION ===')
zs_model = BiEncoder(model_name='bert-base-uncased').to(DEVICE)
zs_corpus_embs = zs_model.encode_docs(corpus_docs, batch_size=8)
zs_retriever   = DenseRetriever(corpus_ids, zs_corpus_embs)
zs_query_embs  = zs_model.encode_queries(query_texts, batch_size=32)

zs_results = zs_retriever.batch_retrieve(zs_query_embs, k=10)
zs_metrics_list = []
for i, row in test_df.iterrows():
    retrieved = [pid for pid, _ in zs_results[i]]
    zs_metrics_list.append(compute_metrics(retrieved, row['product_id']))

zs_agg = aggregate(zs_metrics_list)
print_metrics_table(zs_agg, title='Zero-shot BiEncoder (no fine-tuning)')

In [ ]:
# Fine-tuned evaluation
if TRAINED:
    print('\n=== FINE-TUNED BI-ENCODER EVALUATION ===')
    model = BiEncoder.load(CHECKPOINT)
    model = model.to(DEVICE)
    
    corpus_embs = model.encode_docs(corpus_docs, batch_size=8)
    retriever   = DenseRetriever(corpus_ids, corpus_embs)
    query_embs  = model.encode_queries(query_texts, batch_size=32)
    
    all_results = retriever.batch_retrieve(query_embs, k=10)
    bi_metrics_list = []
    for i, row in test_df.iterrows():
        retrieved = [pid for pid, _ in all_results[i]]
        bi_metrics_list.append(compute_metrics(retrieved, row['product_id']))
    
    bi_agg = aggregate(bi_metrics_list)
    print_metrics_table(bi_agg, title='Fine-tuned BiEncoder (mean pool, random neg)')
else:
    print('Train the model first!')

In [ ]:
# Load BM25 results for comparison
bm25_results_path = '../results/bm25/metrics.json'
if os.path.exists(bm25_results_path):
    with open(bm25_results_path) as f:
        bm25_agg = json.load(f)
    
    metrics_compare = ['ndcg@10', 'recall@10', 'mrr', 'recall@1']
    print('\n=== COMPARISON TABLE ===')
    print(f'{"System":<30} {" ".join(f"{m:>12}" for m in metrics_compare)}')
    print('-' * 80)
    print(f'{"BM25 Okapi":<30} {" ".join(f"{bm25_agg.get(m, 0):>12.4f}" for m in metrics_compare)}')
    print(f'{"Zero-shot BiEncoder":<30} {" ".join(f"{zs_agg.get(m, 0):>12.4f}" for m in metrics_compare)}')
    if TRAINED:
        print(f'{"BiEncoder, mean, random":<30} {" ".join(f"{bi_agg.get(m, 0):>12.4f}" for m in metrics_compare)}')

In [ ]:
# InfoNCE loss demonstration
print('InfoNCE Loss Demonstration:')
print('  sim = q @ p.T / τ   shape (B, B)')
print('  labels = [0, 1, 2, ..., B-1]  (diagonal = positives)')
print('  L = CrossEntropy(sim, labels)')
print()

B = 4
for tau in [0.05, 0.1, 0.5, 1.0]:
    q = torch.nn.functional.normalize(torch.randn(B, 768), dim=-1)
    p = torch.nn.functional.normalize(torch.randn(B, 768), dim=-1)
    loss = infonce_loss(q, p, temperature=tau)
    print(f'  τ={tau:4.2f}: loss={loss.item():.4f} (higher τ → higher initial loss, flatter distribution)')